# 04 — Modelos: Small CNN na imagem e baselines tabulares

Notebook equivalente a `src/models/`. Treina quatro modelos **nas mesmas 5 partições** (`coorte_cv.csv`) e
com **a mesma padronização** (estatística só do treino de cada fold):

| modelo | entrada | o que testa |
|---|---|---|
| `cnn_hilbert` | imagem `(133, 134, 3)` na curva de Hilbert | o formato principal |
| `cnn_vetor` | vetor `(17.818, 3)` na ordem do genoma | a mesma rede em 1D: a dobra 2D ajuda? |
| `logistica` | os 53.454 números do tensor, sem formato | linear, sem imagem |
| `xgboost` | os mesmos 53.454 números | árvores, sem imagem |

Decisões do usuário (2026-10-09): mutação em `log(1 + contagem)` antes de padronizar (feita no `03_images`
e registrada no `padronizacao.json`); rede em **Keras** (motor TensorFlow-CPU); a rede é a **Small CNN** do
projeto irmão; baselines logística e XGBoost já neste notebook.

Entrada: `data/processed/`, gerado pelo `03_images`. Saída: `results/predicoes_oof.csv`, com a probabilidade
de cada classe para cada paciente, prevista pelo modelo do fold em que ele estava no teste. Progresso da execução em
`results/04_models.log` (fold, época e memória). A avaliação
completa (IC por bootstrap, regra do TCGA, ablações por canal) fica para o `05_eval`.

In [1]:
import gc
import json
import os
import time
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # silencia o log C++ do TF (avisos de CPU e de versão do tf.data); erro Python continua aparecendo
import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             matthews_corrcoef)
from sklearn.model_selection import StratifiedShuffleSplit
from xgboost import XGBClassifier

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
PROCESSED, RESULTS = RAIZ / "data" / "processed", RAIZ / "results"
SEED, K = 42, 5
CLASSES = ["CIN", "MSI", "GS", "EBV"]  # = classe_idx 0, 1, 2, 3
CANAIS = ["mutacao", "cnv", "metilacao"]
PROBS = [f"p_{c}" for c in CLASSES]
tf.config.experimental.enable_op_determinism()  # mesma semente, mesmo resultado

# o nbconvert só grava o notebook no fim: o progresso vai para um arquivo (`tail -f results/04_models.log`),
# com a memória do processo, porque a máquina tem 4,8 GB e a 1ª execução morreu por falta de memória
LOG = RESULTS / "04_models.log"


def log(msg, tela=True):
    mem = {l.split(":")[0]: int(l.split()[1]) // 1024 for l in open("/proc/self/status") if l.startswith(("VmRSS", "VmHWM"))}
    linha = f"{time.strftime('%H:%M:%S')} [RAM {mem['VmRSS']} MB, pico {mem['VmHWM']} MB] {msg}"
    with open(LOG, "a") as f:
        f.write(linha + "\n")
    if tela:
        print(linha)


log("=== nova execução ===", tela=False)


coorte = pd.read_csv(PROCESSED / "coorte_cv.csv")
tensor = np.load(PROCESSED / "tensor_bruto.npy")
padronizacao = {int(k): v for k, v in json.loads((PROCESSED / "padronizacao.json").read_text()).items()}
layout = pd.read_csv(PROCESSED / "layout_hilbert.csv")
y, fold = coorte.classe_idx.to_numpy(), coorte.fold.to_numpy()
assert tensor.shape == (len(coorte), 3, len(layout)) and set(padronizacao) == set(range(K))
assert all(s["mutacao"].get("transformacao") == "log1p" for s in padronizacao.values())  # json do 03_images atual
print(f"{len(coorte)} pacientes {dict(zip(CLASSES, np.bincount(y)))} · tensor {tensor.shape} · keras {keras.__version__}")

336 pacientes {'CIN': np.int64(202), 'MSI': np.int64(59), 'GS': np.int64(46), 'EBV': np.int64(29)} · tensor (336, 3, 17818) · keras 3.15.1


## Padronização e imagem — as mesmas funções do `03_images`

Copiadas, porque o notebook é autocontido. A conferência garante que a cópia bate com o `padronizacao.json`:
no treino de cada fold, todo canal fica com média ≈ 0. Se o `log1p` faltasse, a mutação sairia deslocada.

In [2]:
def transformar(x):
    x = x.copy()
    x[:, 0] = np.log1p(x[:, 0])  # mutação em log(1 + contagem), como no 03_images
    return x


def padronizar(x, stats):
    x = transformar(x)
    media = np.array([stats[c]["media"] for c in CANAIS], np.float32)[None, :, None]
    desvio = np.array([stats[c]["desvio"] for c in CANAIS], np.float32)[None, :, None]
    return np.nan_to_num((x - media) / desvio, nan=0.0)


FORMA = (layout.linha.max() + 1, layout.coluna.max() + 1)


def para_hilbert(x):
    img = np.zeros((x.shape[0], x.shape[1], *FORMA), np.float32)
    img[:, :, layout.linha, layout.coluna] = x
    return img


for k in range(K):
    assert np.allclose(padronizar(tensor[fold != k], padronizacao[k]).mean(axis=(0, 2)), 0, atol=1e-3), k
assert FORMA == (133, 134) and not layout.duplicated(["linha", "coluna"]).any()

## Small CNN em Keras

A Small CNN do projeto irmão (`proj_mutacoes_classificacao_TCGA-STAD/src/models/cnn_classifier.py`, em torch),
reescrita em Keras. São 4 blocos *convolução 3 → BatchNorm → ReLU → MaxPool 2*, com 16 → 32 → 64 → 64 filtros,
seguidos de média global, Dropout 0,3 e uma camada densa de 4 saídas. A BatchNorm usa `momentum=0.9`, o
equivalente ao padrão do torch. A mesma função monta a versão 2D
(imagem) e a 1D (vetor). Na 1D, o filtro tem 3 posições em vez de 3×3, por isso ela tem menos parâmetros.

O Keras usa os canais **por último**: a imagem `(3, 133, 134)` entra como `(133, 134, 3)`.

Treino com os hiperparâmetros do irmão:
- **Validação interna:** 15% do treino, estratificada, usada só para decidir quando parar. O fold de
  teste nunca é visto;
- **Parada antecipada:** pelo F1 macro da validação interna, paciência de 15 épocas, até 60 épocas. Os
  pesos voltam para os da melhor época;
- **Otimização:** Adam com lr 1e-3, batch de 32, perda de entropia cruzada com peso inverso à frequência
  da classe;
- **Sem aumento de dados:** girar ou espelhar a imagem tiraria cada gene do seu pixel.

In [3]:
def small_cnn(entrada, n_classes=4, dropout=0.3, filtros=(16, 32, 64, 64)):
    conv, pool, gap = ((layers.Conv2D, layers.MaxPooling2D, layers.GlobalAveragePooling2D) if len(entrada) == 3
                       else (layers.Conv1D, layers.MaxPooling1D, layers.GlobalAveragePooling1D))
    x = entrada_ = keras.Input(entrada)
    for f in filtros:
        x = conv(f, 3, padding="same")(x)
        x = layers.BatchNormalization(momentum=0.9)(x)  # = momentum 0,1 do torch; o 0,99 padrão do Keras demora a assentar
        x = layers.ReLU()(x)
        x = pool(2)(x)
    x = layers.Dropout(dropout)(gap()(x))
    return keras.Model(entrada_, layers.Dense(n_classes)(x))  # logits; a perda aplica o softmax


class F1MacroVal(keras.callbacks.Callback):
    # põe o F1 macro da validação interna nos logs; vem antes do EarlyStopping na lista, que lê o mesmo dicionário
    def __init__(self, x, y):
        super().__init__()
        self.x, self.y, self.f1s = x, y, []

    def on_epoch_end(self, epoch, logs=None):
        self.f1s.append(f1_score(self.y, self.model.predict(self.x, verbose=0).argmax(axis=1), average="macro"))
        logs["val_f1_macro"] = self.f1s[-1]
        log(f"    época {epoch + 1}: perda {logs['loss']:.3f} · F1 macro val {self.f1s[-1]:.3f}", tela=False)


def pesos_por_classe(y_):
    contagem = np.bincount(y_, minlength=len(CLASSES))
    return len(y_) / (len(CLASSES) * contagem)


def treinar_cnn(x_tr, y_tr, x_te, seed):
    keras.backend.clear_session()  # sem isso, cada modelo novo acumula memória no TensorFlow
    gc.collect()
    tr, va = next(StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed).split(x_tr, y_tr))
    keras.utils.set_random_seed(seed)
    modelo = small_cnn(x_tr.shape[1:])
    modelo.compile(keras.optimizers.Adam(1e-3), keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    f1_val = F1MacroVal(x_tr[va], y_tr[va])
    parada = keras.callbacks.EarlyStopping("val_f1_macro", mode="max", patience=15, restore_best_weights=True)
    modelo.fit(x_tr[tr], y_tr[tr], epochs=60, batch_size=32, class_weight=dict(enumerate(pesos_por_classe(y_tr[tr]))),
               callbacks=[f1_val, parada], verbose=0)
    log(f"  {len(f1_val.f1s)} épocas, melhor F1 macro na validação interna {max(f1_val.f1s):.3f}")
    return softmax(modelo.predict(x_te, verbose=0), axis=1)


print({nome: small_cnn(e).count_params() for nome, e in {"2D (Hilbert)": (*FORMA, 3), "1D (vetor)": (len(layout), 3)}.items()})

{'2D (Hilbert)': 61476, '1D (vetor)': 21252}


## Baselines tabulares

Mesma entrada da Small CNN, achatada: os 3 canais × 17.818 genes viram 53.454 colunas, sem formato de
imagem. Configurações iguais às do irmão (`src/eval/baselines.py`):
- **logística:** L2, C = 1, peso balanceado por classe;
- **XGBoost:** 400 árvores, profundidade 4, lr 0,1, subamostra 0,8, metade das colunas por árvore e peso
  inverso à frequência da classe.

In [4]:
def treinar_logistica(x_tr, y_tr, x_te, seed):
    clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, random_state=seed)
    return clf.fit(x_tr, y_tr).predict_proba(x_te)


def treinar_xgboost(x_tr, y_tr, x_te, seed):
    clf = XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.1, subsample=0.8, colsample_bytree=0.5,
                        tree_method="hist", random_state=seed, n_jobs=-1)
    return clf.fit(x_tr, y_tr, sample_weight=pesos_por_classe(y_tr)[y_tr]).predict_proba(x_te)

## Validação cruzada

Para cada fold `k`: padroniza **o tensor inteiro com a estatística do treino de `k`**, treina nos outros 4
folds e prevê o fold `k`. Ao fim, cada paciente tem uma previsão por modelo, feita por um modelo que nunca o
viu.

**Retomada:** cada fold de cada modelo é gravado em `results/oof/` assim que termina. Se a execução cair (a
1ª morreu por falta de memória, com 4,8 GB na máquina), basta rodar de novo: o que já existe é reaproveitado,
e o processo novo começa com a memória zerada. Para treinar tudo de novo, apague `results/oof/`.

In [5]:
MODELOS = {
    "cnn_hilbert": (lambda z: np.moveaxis(para_hilbert(z), 1, -1), treinar_cnn),  # (n, 133, 134, 3)
    "cnn_vetor": (lambda z: np.moveaxis(z, 1, -1), treinar_cnn),                  # (n, 17818, 3)
    "logistica": (lambda z: z.reshape(len(z), -1), treinar_logistica),            # (n, 53454)
    "xgboost": (lambda z: z.reshape(len(z), -1), treinar_xgboost),
}

OOF = RESULTS / "oof"  # uma previsão por modelo e fold, gravada ao terminar; apague a pasta para treinar tudo de novo
OOF.mkdir(exist_ok=True)
pred = []
for nome, (formatar, treinar) in MODELOS.items():
    t0 = time.time()
    for k in range(K):
        arq = OOF / f"{nome}_fold{k}.csv"
        if arq.exists():
            log(f"{nome} · fold {k}: reaproveitado de {arq.name}")
        else:
            log(f"{nome} · fold {k}")
            x = formatar(padronizar(tensor, padronizacao[k]))
            tr, te = fold != k, fold == k
            proba = treinar(x[tr], y[tr], x[te], SEED + k)
            pd.DataFrame(proba, columns=PROBS).assign(
                modelo=nome, amostra=coorte.amostra[te].to_numpy(), fold=k, classe_idx=y[te]).to_csv(arq, index=False)
        pred.append(pd.read_csv(arq))
    log(f"{nome} terminado em {(time.time() - t0) / 60:.1f} min")

pred = pd.concat(pred, ignore_index=True)
assert (pred.groupby("modelo").amostra.nunique() == len(coorte)).all() and len(pred) == len(MODELOS) * len(coorte)
assert np.allclose(pred[PROBS].sum(axis=1), 1, atol=1e-4)
pred.to_csv(RESULTS / "predicoes_oof.csv", index=False)
log("fim: results/predicoes_oof.csv gravado")

12:50:45 [RAM 762 MB, pico 987 MB] cnn_hilbert · fold 0: reaproveitado de cnn_hilbert_fold0.csv
12:50:45 [RAM 762 MB, pico 987 MB] cnn_hilbert · fold 1: reaproveitado de cnn_hilbert_fold1.csv


12:50:45 [RAM 762 MB, pico 987 MB] cnn_hilbert · fold 2: reaproveitado de cnn_hilbert_fold2.csv
12:50:45 [RAM 762 MB, pico 987 MB] cnn_hilbert · fold 3: reaproveitado de cnn_hilbert_fold3.csv
12:50:45 [RAM 762 MB, pico 987 MB] cnn_hilbert · fold 4: reaproveitado de cnn_hilbert_fold4.csv
12:50:45 [RAM 762 MB, pico 987 MB] cnn_hilbert terminado em 0.0 min
12:50:45 [RAM 762 MB, pico 987 MB] cnn_vetor · fold 0: reaproveitado de cnn_vetor_fold0.csv


12:50:45 [RAM 762 MB, pico 987 MB] cnn_vetor · fold 1: reaproveitado de cnn_vetor_fold1.csv
12:50:45 [RAM 762 MB, pico 987 MB] cnn_vetor · fold 2: reaproveitado de cnn_vetor_fold2.csv


12:50:45 [RAM 762 MB, pico 987 MB] cnn_vetor · fold 3: reaproveitado de cnn_vetor_fold3.csv
12:50:45 [RAM 762 MB, pico 987 MB] cnn_vetor · fold 4: reaproveitado de cnn_vetor_fold4.csv
12:50:45 [RAM 762 MB, pico 987 MB] cnn_vetor terminado em 0.0 min
12:50:45 [RAM 762 MB, pico 987 MB] logistica · fold 0: reaproveitado de logistica_fold0.csv
12:50:45 [RAM 762 MB, pico 987 MB] logistica · fold 1: reaproveitado de logistica_fold1.csv


12:50:45 [RAM 762 MB, pico 987 MB] logistica · fold 2: reaproveitado de logistica_fold2.csv


12:50:45 [RAM 762 MB, pico 987 MB] logistica · fold 3: reaproveitado de logistica_fold3.csv
12:50:45 [RAM 762 MB, pico 987 MB] logistica · fold 4: reaproveitado de logistica_fold4.csv
12:50:45 [RAM 762 MB, pico 987 MB] logistica terminado em 0.0 min
12:50:45 [RAM 762 MB, pico 987 MB] xgboost · fold 0: reaproveitado de xgboost_fold0.csv
12:50:45 [RAM 762 MB, pico 987 MB] xgboost · fold 1: reaproveitado de xgboost_fold1.csv
12:50:45 [RAM 762 MB, pico 987 MB] xgboost · fold 2: reaproveitado de xgboost_fold2.csv
12:50:45 [RAM 762 MB, pico 987 MB] xgboost · fold 3: reaproveitado de xgboost_fold3.csv


12:50:45 [RAM 762 MB, pico 987 MB] xgboost · fold 4: reaproveitado de xgboost_fold4.csv


12:50:45 [RAM 762 MB, pico 987 MB] xgboost terminado em 0.0 min
12:50:45 [RAM 764 MB, pico 987 MB] fim: results/predicoes_oof.csv gravado


## Resultado — painel por classe

Métricas sobre as previsões dos 5 folds juntas (336 pacientes; n por classe: CIN 202 · MSI 59 · GS 46 ·
EBV 29). **Pior F1** é o F1 da classe que foi pior: é o número da equidade, o que a pergunta de pesquisa
cobra. O AUC-PR de um modelo que chuta fica na prevalência da classe: CIN 0,60 · MSI 0,18 · GS 0,14 ·
EBV 0,09.

Barra de referência do irmão (só mutação, mesma coorte): regra de marcadores MCC 0,660 · XGBoost 0,655 ·
melhor CNN 0,561; GS nunca passou de F1 ≈ 0,5.

In [6]:
def metricas(d):
    p, yt = d[PROBS].to_numpy(), d.classe_idx.to_numpy()
    yp = p.argmax(axis=1)
    f1 = f1_score(yt, yp, average=None, labels=range(len(CLASSES)))
    return {"MCC": matthews_corrcoef(yt, yp), "acurácia balanceada": balanced_accuracy_score(yt, yp),
            "F1 macro": f1.mean(), "pior F1": f1.min(), **{f"F1 {c}": v for c, v in zip(CLASSES, f1)},
            **{f"AUC-PR {c}": average_precision_score(yt == i, p[:, i]) for i, c in enumerate(CLASSES)}}


pd.DataFrame({m: metricas(d) for m, d in pred.groupby("modelo", sort=False)}).T.round(3)

,MCC,acurácia balanceada,F1 macro,pior F1,F1 CIN,F1 MSI,F1 GS,F1 EBV,AUC-PR CIN,AUC-PR MSI,AUC-PR GS,AUC-PR EBV
cnn_hilbert,0.824,0.913,0.874,0.694,0.907,1.000,0.694,0.897,0.986,1.000,0.830,0.944
cnn_vetor,0.819,0.919,0.874,0.695,0.901,0.983,0.695,0.918,0.979,0.999,0.766,0.988
logistica,0.842,0.916,0.895,0.750,0.924,0.956,0.750,0.951,0.974,0.988,0.788,0.992
xgboost,0.810,0.873,0.872,0.674,0.916,0.930,0.674,0.967,0.972,0.962,0.774,1.000


In [7]:
def confusao(d):
    return pd.DataFrame(confusion_matrix(d.classe_idx, d[PROBS].to_numpy().argmax(axis=1), labels=range(len(CLASSES))),
                        index=pd.Index(CLASSES, name="real"), columns=CLASSES)


pd.concat({m: confusao(d) for m, d in pred.groupby("modelo", sort=False)}, axis=1)  # linhas = real, colunas = previsto

cnn_hilbert             cnn_vetor             logistica              \
             CIN MSI  GS EBV       CIN MSI  GS EBV       CIN MSI  GS EBV   
real                                                                       
CIN          170   0  30   2       169   0  30   3       182   0  18   2   
MSI            0  59   0   0         0  58   1   0         4  54   1   0   
GS             3   0  42   1         4   0  41   1         6   0  39   1   
EBV            0   0   3  26         0   1   0  28         0   0   0  29   

     xgboost              
         CIN MSI  GS EBV  
real                      
CIN      186   2  13   1  
MSI        4  53   2   0  
GS        14   0  31   1  
EBV        0   0   0  29

## Leitura (execução de 2026-10-09; um fold de teste por paciente, uma semente, ainda sem IC)

- **Os quatro modelos ficam na mesma faixa:** MCC 0,81–0,84. O irmão, só com mutação, ficou em 0,66 (regra de
  marcadores) e 0,56 (melhor CNN). CNV e metilação acrescentam muito, como a circularidade previa.
- **A pior classe é GS em todos os modelos:** F1 0,67–0,75. O erro dominante é **CIN previsto como GS**: 30 dos
  202 CIN nas duas Small CNN, 18 na logística, 13 no XGBoost. São CIN de aneuploidia baixa, a fronteira que o TCGA
  traçou pelo grau de aneuploidia.
- **MSI saiu perfeito na Small CNN (Hilbert)** (59/59, AUC-PR 1,000). EBV fica entre 0,90 e 0,97.
- **Hilbert × vetor 1D:** praticamente empatados no F1. A Hilbert tem AUC-PR de GS maior (0,830 × 0,766), ou seja,
  ordena melhor os GS, mas o ponto de corte (argmax) é o mesmo nas duas.
- **A logística, o modelo mais simples, tem o melhor pior-F1 (0,750) e o melhor MCC.** As assinaturas do rótulo
  estão em poucos genes de efeito forte, e um modelo linear as captura.
- Diferenças de 0,02 a 0,05 entre modelos, com GS = 46 e EBV = 29, provavelmente estão dentro do ruído. Quem
  decide é o `05_eval`, com bootstrap pareado, a regra do TCGA e as ablações por canal.
- **Custo:** Small CNN (Hilbert) 10 min, Small CNN (vetor 1D) 25 min, logística 0,6 min, XGBoost 22 min. O XGBoost
  chegou a 3,9 GB de pico, no limite da máquina.

## Conferência: MSI = 1,0 na Small CNN (Hilbert) é sobreajuste?

Pergunta do usuário (2026-10-09). **Sobreajuste** é o modelo decorar o treino e errar em quem não viu. Aqui, toda
métrica vem de pacientes que o modelo **não viu** (fora do fold), então o sobreajuste puxaria o número para
baixo, não para cima. Um 1,0 fora do fold levanta duas outras suspeitas:
1. **Vazamento:** o paciente de teste influenciou o treino de algum jeito;
2. **Tarefa fácil por construção:** a classe se separa por uma variável crua, sem precisar aprender nada.

A célula abaixo confere as duas. Usa só o dado **bruto, sem treino nenhum**: a carga mutacional e o beta do
promotor de MLH1, as duas marcas do MSI no TCGA 2014.

In [8]:
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score

# 1. vazamento: um paciente = uma amostra = um fold de teste (os demais pontos estão conferidos acima e no 03_images)
assert coorte.paciente.is_unique and (pred.groupby("modelo").amostra.nunique() == len(coorte)).all()

# 2. separação por variável crua, sem nenhum modelo
msi = y == 1
crus = {"genes mutados": (tensor[:, 0] > 0).sum(axis=1),               # genes com variante não silenciosa
        "beta MLH1": tensor[:, 2, layout.gene.tolist().index("MLH1")]}  # promotor de MLH1
h = pred[pred.modelo == "cnn_hilbert"]
ic = binomtest(int(msi.sum()), int(msi.sum())).proportion_ci()
print(f"Small CNN (Hilbert), p_MSI: menor entre os MSI {h[h.classe_idx == 1].p_MSI.min():.3f} · "
      f"maior entre os outros {h[h.classe_idx != 1].p_MSI.max():.3f}")
print(f"recall de MSI {msi.sum()}/{msi.sum()}: IC 95% (Clopper-Pearson) {ic.low:.3f}–{ic.high:.3f}")
pd.DataFrame({nome: {"AUC MSI × resto (sem treino)": roc_auc_score(msi, v), "MSI: mínimo": v[msi].min(),
                     "MSI: mediana": np.median(v[msi]), "outros: mediana": np.median(v[~msi]),
                     "outros: máximo": v[~msi].max()} for nome, v in crus.items()}).round(3)

Small CNN (Hilbert), p_MSI: menor entre os MSI 0.492 · maior entre os outros 0.277
recall de MSI 59/59: IC 95% (Clopper-Pearson) 0.939–1.000


,genes mutados,beta MLH1
AUC MSI × resto (sem treino),0.998,0.931
MSI: mínimo,364.000,0.084
MSI: mediana,980.000,0.525
outros: mediana,83.000,0.098
outros: máximo,857.000,0.587


**Leitura:**
- **Não há sinal de vazamento.** Cada paciente está num fold de teste só. A padronização usa só o treino, e a
  parada antecipada usa uma validação tirada do treino. Os hiperparâmetros vieram do irmão, sem ajuste
  olhando o teste.
- **O MSI é quase separável por um número cru.** A contagem de genes mutados, sem modelo nenhum, separa MSI do
  resto com AUC 0,998: mediana de 980 genes no MSI contra 83 nos outros. É a hipermutação, consequência direta
  da instabilidade de microssatélites, que foi o critério do rótulo. Mesmo assim, alguns não MSI chegam a 857
  genes e o MSI mais baixo tem 364: a carga sozinha erra alguns. O MLH1 sozinho fica em AUC 0,931.
- A Small CNN (Hilbert) junta as duas pistas e separa com folga: o MSI menos confiante tem p = 0,49, e o não MSI mais
  confiante, p = 0,28.
- **O 1,0 é esperado pela circularidade, não é sobreajuste.** A barra clínica no MSI também é essencialmente
  perfeita (SKUBLENY 2025, `referencias_literatura.md` §2).
- **Mas 59 de 59 não prova 100%.** O IC 95% do recall vai de 0,939 a 1,000: com outra amostra de pacientes,
  a Small CNN poderia errar 1 ou 2 MSI em 59. As classes difíceis continuam sendo CIN × GS.

# Otimização da Small CNN (2026-10-31)

Decisão do usuário (2026-10-30 e 2026-10-31): otimizar a Small CNN com os itens 1, 2 e 4 da recomendação de
`docs/relatorio_otimizacao.txt` (seção 7), nesta ordem, e ver qual variante fica melhor. Ficam de fora o item 3 (CoordConv,
GAP + GMP, sem o último MaxPool) e o z por gene: no `05_eval` (bloco D), embaralhar os genes nos pixels não mudou nada, e o z
por gene piorou o GS.

O diagnóstico do relatório (seção 1.2): depois da média global, a Small CNN sabe **quanto** de cada padrão há na imagem, mas não
**onde**. Por isso ela não usa o sinal de um gene só (MLH1, CDH1, RHOA), que a logística usa com um peso por gene e canal.

| item | o que é | pergunta |
|---|---|---|
| 1 | pré-registro e BBC-CV | quanto do "ganho" da melhor variante é só a sorte de ter escolhido a melhor? |
| 2 | ramo linear ao lado da Small CNN (*wide & deep*) | dar à rede um peso por pixel recupera o sinal de gene isolado? |
| 4 | AdamW forte, focal loss e group DRO | treinar para o pior caso melhora o GS? |

**Pré-registro** (gravado no diário em 2026-10-31, antes de treinar):
- **métrica principal:** pior F1 no argmax, nas 336 previsões fora do fold, como média das 5 sementes (42, 142, 242, 342, 442;
  no fold `k`, a rede usa `semente + k`). Secundárias: F1 por classe, MCC, AUC-PR do GS e o pior F1 da média das
  probabilidades das 5 sementes (o *ensemble*);
- tudo igual à Small CNN (Hilbert) de cima, salvo a mudança de cada variante;
- **item 2:** `largo` (só o ramo linear: o controle sem convolução), `profundo` (a Small CNN atual, com as sementes já
  gravadas) e `largo_profundo` (os dois somados nos logits). O L2 do ramo linear é 1/(2n), o equivalente ao C = 1 da logística;
- **item 4**, sobre a melhor entre `profundo` e `largo_profundo` (maior média do pior F1; em empate, `profundo`): `adamw`
  (decaimento de peso 1,0), `adamw_focal` (focal loss, γ = 2) e `adamw_dro` (group DRO por classe, η = 0,01);
- **todas as variantes são reportadas**, e a melhor é corrigida pela BBC-CV.

**Retomada:** cada fold de cada variante e semente vai para `results/oof_otim/` assim que termina. O `profundo` e a Small CNN
(vetor 1D) reaproveitam as sementes já gravadas (`results/oof/` para a semente 42; `results/oof_b/`, do bloco B do `05_eval`,
para as outras). Se faltarem, são treinadas aqui. Para treinar tudo de novo, apague `results/oof_otim/`.

Custo previsto (8 CPUs, sem GPU): 25 rodadas de 5 folds. São 5 do `largo`, de poucos minutos cada, e 20 do `largo_profundo` e do
item 4, de 10 a 15 min cada, como a Small CNN. Total ≈ 4 a 5 h. Progresso em `results/04_models.log`.

## Item 1 — avaliação honesta: pré-registro e BBC-CV

**O problema.** Treinar várias variantes e ficar com a melhor nos mesmos 336 pacientes infla o resultado. Já aconteceu aqui: com
o limiar de GS, a melhor das 5 sementes deu 0,771 de pior F1, e a média das 5 ficou em 0,669. A conferência abaixo mostra o
tamanho do efeito: entre 20 configurações **de mesma qualidade** (85% de acerto ao acaso), a maior fica bem acima da média
sem ser melhor em nada.

**A primeira proteção é o pré-registro:** as variantes, a métrica e a regra de escolha foram escritas no diário antes do treino,
e todas são reportadas.

**A segunda é a BBC-CV** (*bootstrap bias corrected CV*; TSAMARDINOS et al., 2018; relatório, seção 2.3). Ela usa só as
previsões fora do fold que já existem, sem treinar nada a mais. Em cada um dos 2.000 sorteios do bootstrap:
1. calcula o pior F1 de cada configuração nos pacientes **sorteados**;
2. escolhe a melhor delas;
3. mede essa escolhida nos pacientes que **ficaram de fora** do sorteio (~37% dos 336).

A média do passo 3 estima o desempenho do procedimento inteiro, "testar várias e ficar com a melhor", e não o da vencedora com
sorte. Os sorteios são os mesmos 2.000 do `05_eval` (estratificados: cada classe mantém o seu n).

In [9]:
# ---- cópias do 05_eval: matriz de confusão e painel, vetorizados nos sorteios do bootstrap ----
GS = CLASSES.index("GS")
SEMENTES = [SEED, 142, 242, 342, 442]


def confusoes(y_, yp, idx=None):
    # matrizes de confusão (linha = real, coluna = previsto); com idx (B, n), uma por reamostragem, sem laço
    idx = np.arange(len(y_))[None] if idx is None else idx
    cod = y_[idx] * 4 + yp[idx] + 16 * np.arange(len(idx))[:, None]
    return np.bincount(cod.ravel(), minlength=16 * len(idx)).reshape(-1, 4, 4)


def painel(cm):
    # cm (..., 4, 4) → métricas com forma (...); MCC multiclasse de Gorodkin, o mesmo do scikit-learn
    vp = np.diagonal(cm, axis1=-2, axis2=-1)
    real, prev = cm.sum(-1), cm.sum(-2)
    s, c = cm.sum((-2, -1)), vp.sum(-1)
    f1 = 2 * vp / np.maximum(real + prev, 1)
    mcc = (c * s - (prev * real).sum(-1)) / np.sqrt((s**2 - (prev**2).sum(-1)) * (s**2 - (real**2).sum(-1)))
    return {"MCC": mcc, "acurácia balanceada": (vp / real).mean(-1), "F1 macro": f1.mean(-1), "pior F1": f1.min(-1),
            **{f"F1 {k}": f1[..., i] for i, k in enumerate(CLASSES)}}


rng = np.random.default_rng(SEED)  # os mesmos 2.000 sorteios estratificados do 05_eval
idx = np.concatenate([rng.choice(np.flatnonzero(y == c), (2000, (y == c).sum())) for c in range(4)], axis=1)


def bbc_cv(preds):
    # BBC-CV: em cada sorteio, escolhe a configuração de maior pior F1 nos pacientes sorteados e mede essa escolha nos que
    # ficaram de fora. preds: {nome: classe prevista (336,)}
    vezes = np.stack([np.bincount(i, minlength=len(y)) for i in idx])  # (2.000, 336): vezes que cada paciente saiu
    dentro, fora = [], []
    for yp in preds.values():
        celula = np.eye(16, dtype=int)[y * 4 + yp]                        # (336, 16): célula da matriz de cada paciente
        dentro.append(painel((vezes @ celula).reshape(-1, 4, 4))["pior F1"])
        fora.append(painel(((vezes == 0) @ celula).reshape(-1, 4, 4))["pior F1"])
    escolha = np.argmax(dentro, axis=0)
    v = np.asarray(fora)[escolha, np.arange(len(idx))]
    return {"maior pior F1 (ingênuo)": max(float(painel(confusoes(y, yp)[0])["pior F1"]) for yp in preds.values()),
            "BBC-CV": v.mean(), "IC 2,5%": np.percentile(v, 2.5), "IC 97,5%": np.percentile(v, 97.5),
            "mais escolhida": list(preds)[np.bincount(escolha).argmax()],
            "% dos sorteios": 100 * np.bincount(escolha).max() / len(idx)}


# autoteste 1: a matriz pela contagem de sorteios é a mesma do confusoes() do 05_eval
rt = np.random.default_rng(0)
yp_t = np.where(rt.random(len(y)) < 0.85, y, rt.integers(0, 4, len(y)))
vezes = np.stack([np.bincount(i, minlength=len(y)) for i in idx[:50]])
assert np.array_equal((vezes @ np.eye(16, dtype=int)[y * 4 + yp_t]).reshape(-1, 4, 4), confusoes(y, yp_t, idx[:50]))
# autoteste 2: 20 configurações de mesma qualidade (85% de acerto ao acaso): a maior é otimista, a BBC-CV não
ruido = {i: np.where(rt.random(len(y)) < 0.85, y, rt.integers(0, 4, len(y))) for i in range(20)}
r = bbc_cv(ruido)
media = np.mean([painel(confusoes(y, p)[0])["pior F1"] for p in ruido.values()])
assert r["maior pior F1 (ingênuo)"] > media + 0.04 and abs(r["BBC-CV"] - media) < 0.04, (r, media)
print(f"20 configurações iguais: média {media:.3f} · maior {r['maior pior F1 (ingênuo)']:.3f} · BBC-CV {r['BBC-CV']:.3f}")

20 configurações iguais: média 0.790 · maior 0.877 · BBC-CV 0.787


### As variantes, em código

Uma função de treino só, `treinar_variante()`: é o `treinar_cnn()` de cima, com as mudanças ligáveis. Com tudo desligado, é a
Small CNN atual.

- **Ramo largo** (*wide & deep*; relatório, seção 4.1): uma camada densa direto sobre os 53.466 valores da imagem (133 × 134
  pixels × 3 canais), somada aos logits da Small CNN. É uma logística dentro da rede: cada gene e canal ganha o seu peso, e o
  pixel do MLH1 volta a ter identidade. O L2 de 1/(2n) reproduz o C = 1 da logística de cima. No scikit-learn, a penalidade é
  ‖w‖²/2 somada à perda **total**; dividindo tudo pelo n, fica ‖w‖²/(2n) somada à perda **média**, que é a do Keras.
- **AdamW** (relatório, seção 5.7): o Adam com o decaimento de peso separado do gradiente. No Keras, a cada passo cada peso perde
  lr × wd = 0,001 × 1,0 = 0,1% do valor. Em 480 passos (60 épocas de ~8 lotes), sem gradiente nenhum, sobrariam 62%. Bias e
  BatchNorm não decaem.
- **Focal loss** (LIN et al., 2018; relatório, seção 5.2): multiplica a perda de cada paciente por (1 − p)^γ, com p a
  probabilidade da classe certa. Com γ = 2, um paciente com p = 0,9 passa a pesar 1% do que pesava, e um com p = 0,4, 36%.
- **Group DRO** (SAGAWA et al., 2020; relatório, seção 5.4), com grupo = classe. A cada lote, o peso q de cada classe é
  multiplicado por exp(η × perda média da classe no lote) e renormalizado, e a perda do lote é Σ q × perda da classe. A classe
  que vai pior ganha peso aos poucos. Substitui o peso de classe fixo. Pelo relatório, o group DRO só funciona com regularização
  forte, por isso ele entra junto com o AdamW.

As duas conferências no fim da célula: o group DRO num lote de 4 pacientes feito à mão, e 1 época de cada variante em 40
pacientes, para que um erro de código apareça em segundos, e não depois de horas de treino.

In [10]:
def rede(entrada, largo=False, profundo=True, l2=0.0):
    # wide & deep: logits da Small CNN + logits de uma densa sobre todos os pixels (o ramo largo é uma logística)
    x = keras.Input(entrada)
    ramos = [small_cnn(entrada)(x)] if profundo else []
    if largo:
        ramos.append(layers.Dense(len(CLASSES), kernel_regularizer=keras.regularizers.L2(l2))(layers.Flatten()(x)))
    return keras.Model(x, ramos[0] if len(ramos) == 1 else layers.Add()(ramos))


def focal(y_true, logits):
    # focal loss com γ = 2 e α = 1 (o α padrão do Keras, 0,25, só mudaria a escala da perda contra o L2)
    um = tf.one_hot(tf.cast(tf.reshape(y_true, [-1]), tf.int32), len(CLASSES))
    return keras.losses.categorical_focal_crossentropy(um, logits, alpha=1.0, gamma=2.0, from_logits=True)


class GroupDRO(keras.losses.Loss):
    # group DRO online, grupo = classe: q_g ← q_g · exp(η · perda média da classe g no lote); perda = Σ_g q_g · perda_g
    def __init__(self, eta=0.01):
        super().__init__(reduction="sum")
        self.eta = eta
        self.q = tf.Variable(tf.fill([len(CLASSES)], 1 / len(CLASSES)), trainable=False)

    def call(self, y_true, logits):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        ce = tf.nn.sparse_softmax_cross_entropy_with_logits(y_true, logits)
        g = tf.one_hot(y_true, len(CLASSES))                # (lote, 4)
        n = tf.maximum(tf.reduce_sum(g, 0), 1.0)
        perda_g = tf.reduce_sum(g * ce[:, None], 0) / n     # classe ausente do lote: perda 0, q dela não sobe
        q = self.q.assign(self.q * tf.exp(self.eta * tf.stop_gradient(perda_g)))
        q = self.q.assign(q / tf.reduce_sum(q))
        return ce * tf.reduce_sum(g * q / n, 1)             # somado no lote = Σ_g q_g · perda_g


def treinar_variante(x_tr, y_tr, x_te, seed, largo=False, profundo=True, adamw=False, perda="ce", epocas=60):
    # = treinar_cnn(), com as mudanças pré-registradas; com tudo desligado, é a Small CNN atual
    keras.backend.clear_session()
    gc.collect()
    tr, va = next(StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed).split(x_tr, y_tr))
    keras.utils.set_random_seed(seed)
    modelo = rede(x_tr.shape[1:], largo, profundo, l2=1 / (2 * len(tr)))  # L2 do ramo largo = C 1 da logística
    otimizador = keras.optimizers.AdamW(1e-3, weight_decay=1.0) if adamw else keras.optimizers.Adam(1e-3)
    if adamw:
        otimizador.exclude_from_weight_decay(var_names=["bias", "gamma", "beta"])  # só os kernels decaem
    funcao = (GroupDRO() if perda == "dro" else focal if perda == "focal"
              else keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    modelo.compile(otimizador, funcao)
    f1_val = F1MacroVal(x_tr[va], y_tr[va])
    parada = keras.callbacks.EarlyStopping("val_f1_macro", mode="max", patience=15, restore_best_weights=True)
    pesos = None if perda == "dro" else dict(enumerate(pesos_por_classe(y_tr[tr])))  # o DRO substitui o peso de classe
    modelo.fit(x_tr[tr], y_tr[tr], epochs=epocas, batch_size=32, class_weight=pesos, callbacks=[f1_val, parada], verbose=0)
    log(f"  {len(f1_val.f1s)} épocas, melhor F1 macro na validação interna {max(f1_val.f1s):.3f}")
    return softmax(modelo.predict(x_te, verbose=0), axis=1)


OOF_O = RESULTS / "oof_otim"  # uma previsão por variante, semente e fold; apague a pasta para treinar de novo
OOF_O.mkdir(exist_ok=True)
FORMATOS = {"hilbert": lambda z: np.moveaxis(para_hilbert(z), 1, -1), "vetor": lambda z: np.moveaxis(z, 1, -1)}
ITEM2 = {"largo": {"largo": True, "profundo": False}, "largo_profundo": {"largo": True}}
ITEM4 = {"adamw": {"adamw": True}, "adamw_focal": {"adamw": True, "perda": "focal"},
         "adamw_dro": {"adamw": True, "perda": "dro"}}


def gravados(modelo, s):
    # molde do arquivo de uma semente já treinada da Small CNN atual: 04_models (semente 42) ou bloco B do 05_eval
    return str(OOF / f"{modelo}_fold{{k}}.csv") if s == SEED else str(RESULTS / "oof_b" / f"{modelo}_s{s}_fold{{k}}.csv")


def rodar_o(nome, semente, cfg, formato="hilbert", molde=None):
    # previsões fora do fold (336, 4) na ordem do coorte_cv.csv; reaproveita o que já existe e treina o resto
    proba = np.empty((len(y), len(CLASSES)))
    for k in range(K):
        te = fold == k
        arq = Path(molde.format(k=k)) if molde and Path(molde.format(k=k)).exists() else OOF_O / f"{nome}_s{semente}_fold{k}.csv"
        if not arq.exists():
            log(f"{nome} · semente {semente} · fold {k}")
            x = FORMATOS[formato](padronizar(tensor, padronizacao[k]))
            p = treinar_variante(x[~te], y[~te], x[te], semente + k, **cfg)
            pd.DataFrame(p, columns=PROBS).assign(amostra=coorte.amostra[te].to_numpy()).to_csv(arq, index=False)
            del x
        proba[te] = pd.read_csv(arq).set_index("amostra").loc[coorte.amostra[te], PROBS].to_numpy()
    return proba


# autoteste do group DRO: num lote em que a classe 1 vai mal, o peso dela sobe, e a perda é Σ q_g · perda_g
dro = GroupDRO(eta=1.0)
yt, lg = tf.constant([0, 0, 1, 1]), tf.constant([[3.0, 0, 0, 0]] * 4)
total, q = float(dro(yt, lg)), dro.q.numpy()
ce = tf.nn.sparse_softmax_cross_entropy_with_logits(yt, lg).numpy()
assert q[1] > q[0] > q[2] == q[3] and np.isclose(q.sum(), 1)
assert np.isclose(total, q[0] * ce[:2].mean() + q[1] * ce[2:].mean(), rtol=1e-5)

# autoteste das variantes: 1 época em 40 pacientes de treino do fold 0, para pegar erro antes das horas de treino
sub = np.concatenate([np.flatnonzero((y == c) & (fold != 0))[:10] for c in range(4)])
x = FORMATOS["hilbert"](padronizar(tensor[sub], padronizacao[0]))
for nome, cfg in {"profundo": {}, **ITEM2, **ITEM4, "largo_profundo + adamw_dro": ITEM2["largo_profundo"] | ITEM4["adamw_dro"]}.items():
    p = treinar_variante(x, y[sub], x[:3], SEED, **cfg, epocas=1)
    assert p.shape == (3, 4) and np.allclose(p.sum(axis=1), 1, atol=1e-4), nome
del x
print({nome: rede((*FORMA, 3), **{k: v for k, v in cfg.items() if k in ("largo", "profundo")}).count_params()
       for nome, cfg in {"profundo": {}, **ITEM2}.items()})

12:50:53 [RAM 1076 MB, pico 1190 MB]   1 épocas, melhor F1 macro na validação interna 0.071


12:50:55 [RAM 953 MB, pico 1190 MB]   1 épocas, melhor F1 macro na validação interna 0.071


12:51:02 [RAM 1157 MB, pico 1312 MB]   1 épocas, melhor F1 macro na validação interna 0.071


12:51:09 [RAM 1301 MB, pico 1406 MB]   1 épocas, melhor F1 macro na validação interna 0.071


12:51:15 [RAM 1312 MB, pico 1436 MB]   1 épocas, melhor F1 macro na validação interna 0.071


12:51:21 [RAM 1355 MB, pico 1459 MB]   1 épocas, melhor F1 macro na validação interna 0.071


12:51:28 [RAM 1184 MB, pico 1476 MB]   1 épocas, melhor F1 macro na validação interna 0.071


{'profundo': 61476, 'largo': 213868, 'largo_profundo': 275344}


### Referência estável: as 5 sementes que já existem

Antes de qualquer variante, a Small CNN (Hilbert) atual (`profundo`) e a Small CNN (vetor 1D), com as 5 sementes já gravadas e
sem treino. Duas formas de resumir: a **média do pior F1** das 5 execuções (a métrica pré-registrada) e o pior F1 do **ensemble**
(a média das probabilidades das 5, uma previsão só). Embaixo, a BBC-CV nas 10 execuções: quanto vale, de verdade, "ficar com a
melhor semente".

In [11]:
PO = {}  # (variante, semente) → probabilidades fora do fold (336, 4)
for s in SEMENTES:
    PO[("profundo", s)] = rodar_o("profundo", s, {}, molde=gravados("cnn_hilbert", s))
    PO[("vetor 1D", s)] = rodar_o("vetor", s, {}, "vetor", gravados("cnn_vetor", s))
P_LOG = pred[pred.modelo == "logistica"].set_index("amostra").loc[coorte.amostra, PROBS].to_numpy()


def resumo(variantes):
    # por variante: média e desvio das 5 sementes e o pior F1 do ensemble; a logística (determinística) como referência
    linhas = {}
    for v in variantes:
        ps = [PO[(v, s)] for s in SEMENTES]
        m = pd.DataFrame([{k: float(x) for k, x in painel(confusoes(y, p.argmax(axis=1))[0]).items()} for p in ps])
        linhas[v] = {"pior F1": m["pior F1"].mean(), "dp": m["pior F1"].std(), "mín.": m["pior F1"].min(),
                     "máx.": m["pior F1"].max(), **{f"F1 {c}": m[f"F1 {c}"].mean() for c in CLASSES}, "MCC": m.MCC.mean(),
                     "AUC-PR GS": np.mean([average_precision_score(y == GS, p[:, GS]) for p in ps]),
                     "pior F1 do ensemble": float(painel(confusoes(y, np.mean(ps, axis=0).argmax(axis=1))[0])["pior F1"])}
    m = {k: float(x) for k, x in painel(confusoes(y, P_LOG.argmax(axis=1))[0]).items()}
    linhas["logística (1 execução)"] = {"pior F1": m["pior F1"], **{f"F1 {c}": m[f"F1 {c}"] for c in CLASSES},
                                       "MCC": m["MCC"], "AUC-PR GS": average_precision_score(y == GS, P_LOG[:, GS])}
    return pd.DataFrame(linhas).T


display(resumo(["profundo", "vetor 1D"]).round(3))
pd.Series(bbc_cv({f"{v} s{s}": PO[(v, s)].argmax(axis=1) for v in ["profundo", "vetor 1D"] for s in SEMENTES}),
          name="ficar com a melhor das 10 execuções").to_frame()

,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble
profundo,0.688,0.025,0.660,0.722,0.919,0.977,0.688,0.892,0.821,0.748,0.772
vetor 1D,0.712,0.016,0.695,0.737,0.909,0.978,0.712,0.892,0.823,0.772,0.730
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN


,ficar com a melhor das 10 execuções
maior pior F1 (ingênuo),0.736842
BBC-CV,0.695636
"IC 2,5%",0.551724
"IC 97,5%",0.818182
mais escolhida,vetor 1D s442
% dos sorteios,46.85


## Item 2 — ramo linear ao lado da Small CNN (*wide & deep*)

Duas variantes novas, 5 sementes cada (o `profundo` já está acima, sem retreino):
- `largo`: só o ramo linear, sem convolução. É o controle: uma logística treinada como a rede (Adam, parada antecipada, peso de
  classe), para separar o efeito do ramo do efeito do treino;
- `largo_profundo`: os dois ramos somados nos logits.

Como ler: se `largo_profundo` > `profundo`, o peso por pixel ajuda a Small CNN. Se `largo_profundo` ≈ `largo`, quem trabalha é o
ramo linear, e a convolução não acrescenta nada. É o que pergunta a pendência "a imagem é necessária?".

In [12]:
t0 = time.time()
for v, cfg in ITEM2.items():
    for s in SEMENTES:
        log(f"item 2: {v}, semente {s}")
        PO[(v, s)] = rodar_o(v, s, cfg)
log(f"item 2 terminado em {(time.time() - t0) / 60:.0f} min")
resumo(["profundo", *ITEM2]).round(3)

12:51:31 [RAM 1197 MB, pico 1476 MB] item 2: largo, semente 42
12:51:31 [RAM 1197 MB, pico 1476 MB] largo · semente 42 · fold 0


12:51:40 [RAM 1551 MB, pico 1644 MB]   21 épocas, melhor F1 macro na validação interna 0.866


12:51:40 [RAM 1446 MB, pico 1644 MB] largo · semente 42 · fold 1


12:51:49 [RAM 1589 MB, pico 1691 MB]   21 épocas, melhor F1 macro na validação interna 0.943


12:51:49 [RAM 1481 MB, pico 1691 MB] largo · semente 42 · fold 2


12:51:56 [RAM 1611 MB, pico 1722 MB]   18 épocas, melhor F1 macro na validação interna 0.885
12:51:57 [RAM 1498 MB, pico 1722 MB] largo · semente 42 · fold 3


... (83 outputs left out)

13:45:58 [RAM 2240 MB, pico 2933 MB] largo_profundo · semente 442 · fold 4


13:48:10 [RAM 2649 MB, pico 2944 MB]   18 épocas, melhor F1 macro na validação interna 0.824


13:48:10 [RAM 2541 MB, pico 2944 MB] item 2 terminado em 57 min


,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble
profundo,0.688,0.025,0.660,0.722,0.919,0.977,0.688,0.892,0.821,0.748,0.772
largo,0.710,0.051,0.642,0.760,0.911,0.933,0.710,0.921,0.807,0.710,0.714
largo_profundo,0.667,0.070,0.558,0.733,0.909,0.922,0.667,0.876,0.787,0.686,0.701
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN


## Item 4 — perda e regularização para o pior caso

Sobre a melhor entre `profundo` e `largo_profundo` (pré-registrado: maior média do pior F1; em empate, `profundo`). O `largo` não
entra: ele não tem convolução, e a pergunta é otimizar a Small CNN. Três variantes, 5 sementes cada:
- `adamw`: só o decaimento de peso forte. É a referência das outras duas;
- `adamw_focal`: o mesmo, com focal loss;
- `adamw_dro`: o mesmo, com group DRO.

Como ler: `adamw` contra a base mede a regularização; `adamw_focal` e `adamw_dro` contra `adamw` medem a perda.

In [13]:
medias = {v: np.mean([painel(confusoes(y, PO[(v, s)].argmax(axis=1))[0])["pior F1"] for s in SEMENTES])
          for v in ["profundo", "largo_profundo"]}
BASE4 = "largo_profundo" if medias["largo_profundo"] > medias["profundo"] else "profundo"  # em empate, profundo
log(f"item 4 sobre {BASE4} (pior F1 médio: profundo {medias['profundo']:.3f}, largo_profundo {medias['largo_profundo']:.3f})")
t0 = time.time()
for v, cfg in ITEM4.items():
    for s in SEMENTES:
        log(f"item 4: {BASE4} + {v}, semente {s}")
        PO[(f"{BASE4} + {v}", s)] = rodar_o(f"{BASE4}_{v}", s, ITEM2.get(BASE4, {}) | cfg)
log(f"item 4 terminado em {(time.time() - t0) / 60:.0f} min")
resumo([BASE4, *(f"{BASE4} + {v}" for v in ITEM4)]).round(3)

13:48:10 [RAM 2541 MB, pico 2944 MB] item 4 sobre profundo (pior F1 médio: profundo 0.688, largo_profundo 0.667)
13:48:10 [RAM 2541 MB, pico 2944 MB] item 4: profundo + adamw, semente 42


13:48:11 [RAM 2541 MB, pico 2944 MB] profundo_adamw · semente 42 · fold 0


13:50:28 [RAM 2675 MB, pico 2971 MB]   19 épocas, melhor F1 macro na validação interna 0.840


13:50:29 [RAM 2564 MB, pico 2971 MB] profundo_adamw · semente 42 · fold 1


13:52:17 [RAM 2704 MB, pico 2995 MB]   19 épocas, melhor F1 macro na validação interna 0.918


13:52:18 [RAM 2583 MB, pico 2995 MB] profundo_adamw · semente 42 · fold 2


... (143 outputs left out)

16:55:52 [RAM 3251 MB, pico 3673 MB] profundo_adamw_dro · semente 442 · fold 4


16:58:09 [RAM 3401 MB, pico 3683 MB]   31 épocas, melhor F1 macro na validação interna 0.848


16:58:10 [RAM 3285 MB, pico 3683 MB] item 4 terminado em 190 min


,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble
profundo,0.688,0.025,0.660,0.722,0.919,0.977,0.688,0.892,0.821,0.748,0.772
profundo + adamw,0.691,0.022,0.661,0.714,0.919,0.970,0.691,0.878,0.816,0.776,0.750
profundo + adamw_focal,0.692,0.011,0.682,0.710,0.914,0.969,0.692,0.932,0.821,0.739,0.740
profundo + adamw_dro,0.717,0.015,0.692,0.733,0.926,0.962,0.717,0.922,0.834,0.778,0.757
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN


## Comparação final

Todas as variantes pré-registradas, nas 336 previsões fora do fold:
1. o resumo de cada uma (média das 5 sementes e ensemble), gravado em `results/04_models_otimizacao.csv`;
2. a diferença pareada contra o `profundo` (mesma semente) e contra a logística, como média das 5 sementes, com IC 95% pelo
   bootstrap pareado (os mesmos 2.000 sorteios do `05_eval`);
3. a BBC-CV sobre as 6 configurações (o ensemble de cada uma) e sobre as 30 execuções: o valor honesto de "ficar com a melhor".

In [14]:
VARIANTES = list(dict.fromkeys(v for v, _ in PO if v != "vetor 1D"))
tab_o = resumo(VARIANTES)
tab_o.to_csv(RESULTS / "04_models_otimizacao.csv")
display(tab_o.round(3))


def dif_media(pares, met):
    # média, sobre as sementes, da diferença a − b; IC 95% pelo bootstrap pareado (cópia do 05_eval, bloco D)
    pont = np.mean([painel(confusoes(y, a)[0])[met] - painel(confusoes(y, b)[0])[met] for a, b in pares])
    lo, hi = np.percentile(np.mean([painel(confusoes(y, a, idx))[met] - painel(confusoes(y, b, idx))[met]
                                    for a, b in pares], axis=0), [2.5, 97.5])
    return f"{pont:+.3f} ({lo:+.3f} a {hi:+.3f})"


MET = ["pior F1", "F1 GS", "F1 EBV", "MCC"]
comparacoes = {}
for v in VARIANTES:
    a = [PO[(v, s)].argmax(axis=1) for s in SEMENTES]
    if v != "profundo":
        b = [PO[("profundo", s)].argmax(axis=1) for s in SEMENTES]
        comparacoes[f"{v} − profundo"] = {m: dif_media(list(zip(a, b)), m) for m in MET}
    comparacoes[f"{v} − logística"] = {m: dif_media([(p, P_LOG.argmax(axis=1)) for p in a], m) for m in MET}
print("Diferença (média das 5 sementes) e IC 95%:")
display(pd.DataFrame(comparacoes).T)

ens = {v: np.mean([PO[(v, s)] for s in SEMENTES], axis=0).argmax(axis=1) for v in VARIANTES}
pd.DataFrame({"6 configurações (ensemble de cada)": bbc_cv(ens),
              "30 execuções": bbc_cv({f"{v} s{s}": PO[(v, s)].argmax(axis=1) for v in VARIANTES for s in SEMENTES})})

,pior F1,dp,mín.,máx.,F1 CIN,F1 MSI,F1 GS,F1 EBV,MCC,AUC-PR GS,pior F1 do ensemble
profundo,0.688,0.025,0.660,0.722,0.919,0.977,0.688,0.892,0.821,0.748,0.772
largo,0.710,0.051,0.642,0.760,0.911,0.933,0.710,0.921,0.807,0.710,0.714
largo_profundo,0.667,0.070,0.558,0.733,0.909,0.922,0.667,0.876,0.787,0.686,0.701
profundo + adamw,0.691,0.022,0.661,0.714,0.919,0.970,0.691,0.878,0.816,0.776,0.750
profundo + adamw_focal,0.692,0.011,0.682,0.710,0.914,0.969,0.692,0.932,0.821,0.739,0.740
profundo + adamw_dro,0.717,0.015,0.692,0.733,0.926,0.962,0.717,0.922,0.834,0.778,0.757
logística (1 execução),0.750,NaN,NaN,NaN,0.924,0.956,0.750,0.951,0.842,0.788,NaN


Diferença (média das 5 sementes) e IC 95%:


,pior F1,F1 GS,F1 EBV,MCC
profundo − logística,-0.062 (-0.114 a -0.011),-0.062 (-0.114 a -0.011),-0.059 (-0.105 a -0.019),-0.022 (-0.055 a +0.011)
largo − profundo,+0.022 (-0.023 a +0.068),+0.022 (-0.023 a +0.068),+0.029 (-0.014 a +0.072),-0.014 (-0.046 a +0.017)
largo − logística,-0.040 (-0.087 a +0.002),-0.040 (-0.087 a +0.002),-0.030 (-0.054 a -0.012),-0.035 (-0.060 a -0.011)
largo_profundo − profundo,-0.021 (-0.080 a +0.037),-0.021 (-0.080 a +0.037),-0.016 (-0.054 a +0.033),-0.034 (-0.069 a +0.001)
largo_profundo − logística,-0.083 (-0.144 a -0.033),-0.083 (-0.144 a -0.033),-0.074 (-0.112 a -0.040),-0.055 (-0.085 a -0.028)
profundo + adamw − profundo,+0.003 (-0.026 a +0.032),+0.003 (-0.026 a +0.032),-0.014 (-0.043 a +0.011),-0.005 (-0.019 a +0.010)
profundo + adamw − logística,-0.059 (-0.109 a -0.011),-0.059 (-0.109 a -0.011),-0.073 (-0.135 a -0.018),-0.026 (-0.061 a +0.007)
profundo + adamw_focal − profundo,+0.004 (-0.016 a +0.026),+0.004 (-0.016 a +0.026),+0.040 (+0.011 a +0.073),+0.000 (-0.013 a +0.014)
profundo + adamw_focal − logística,-0.058 (-0.111 a -0.008),-0.058 (-0.111 a -0.008),-0.019 (-0.060 a +0.018),-0.021 (-0.053 a +0.011)
profundo + adamw_dro − profundo,+0.030 (+0.003 a +0.058),+0.030 (+0.002 a +0.058),+0.030 (-0.005 a +0.067),+0.013 (-0.005 a +0.031)


,6 configurações (ensemble de cada),30 execuções
maior pior F1 (ingênuo),0.772277,0.76
BBC-CV,0.751472,0.704823
"IC 2,5%",0.604535,0.555556
"IC 97,5%",0.869565,0.823529
mais escolhida,profundo,largo s242
% dos sorteios,80.25,36.05


### Leitura da otimização

Pré-registro no diário (2026-10-31). Métrica principal: pior F1 no argmax, média das 5 sementes, nas 336 previsões fora do
fold (GS = 46, EBV = 29). Em todas as variantes, e em todas as sementes, a classe que cai é o **GS**: o pior F1 é o F1 do GS.

| variante | pior F1 (média ± dp) | MCC | AUC-PR GS |
|---|---|---|---|
| `profundo` (Small CNN (Hilbert) atual) | 0,688 ± 0,025 | 0,821 | 0,748 |
| `largo` (só o ramo linear, sem convolução) | 0,710 ± 0,051 | 0,807 | 0,710 |
| `largo_profundo` (wide & deep) | 0,667 ± 0,070 | 0,787 | 0,686 |
| `profundo` + AdamW | 0,691 ± 0,022 | 0,816 | 0,776 |
| `profundo` + AdamW + focal loss | 0,692 ± 0,011 | 0,821 | 0,739 |
| `profundo` + AdamW + group DRO | **0,717 ± 0,015** | 0,834 | 0,778 |
| logística (1 execução) | 0,750 | 0,842 | 0,788 |

1. **O wide & deep não ajudou.** O `largo_profundo` ficou abaixo do `profundo` (−0,021, IC −0,080 a +0,037), com o maior dp.
   Pela regra pré-registrada, a base do item 4 foi o `profundo`.
2. **A melhor variante é AdamW + group DRO** (0,717 ± 0,015):
   - contra o `profundo`: +0,030 (IC +0,003 a +0,058). O IC exclui o 0 por pouco, e sem correção para as 5 comparações
     contra o `profundo`. No MCC, +0,013 (IC −0,005 a +0,031);
   - contra a logística: −0,033 (IC −0,081 a +0,013). Não a alcança;
   - AdamW sozinho (+0,003) e com focal loss (+0,004) não mudam o pior F1. A focal loss sobe o EBV (+0,040, IC +0,011 a
     +0,073), mas o GS fica igual.
3. **BBC-CV** (corrige o otimismo de ficar com a melhor):
   - nas 30 execuções: a melhor execução isolada dá 0,760; corrigida, **0,705** (IC 0,556 a 0,824). A mais escolhida nos
     sorteios é uma semente do `largo` (36%), não o DRO;
   - nas 6 configurações (média das probabilidades das 5 sementes): 0,772 → **0,751** (IC 0,605 a 0,870). A mais escolhida
     é o `profundo` (80%), porque o ensemble do DRO (0,757) fica abaixo do ensemble do `profundo` (0,772);
   - leitura: o "ganho" de escolher a melhor execução some com a correção (0,705 está na faixa das variantes). O ganho do
     DRO na média das sementes não aparece no ensemble. O 0,751 corrigido empata com a logística (0,750), mas compara um
     ensemble de 5 redes com uma execução da logística, e não é a métrica principal do pré-registro.
4. **A imagem é necessária?** O `largo` é uma camada densa sobre os mesmos 53.466 valores da imagem, sem convolução. Ele
   fica +0,022 acima do `profundo` (IC −0,023 a +0,068): a convolução não acrescenta nada mensurável a um modelo linear nos
   mesmos valores. E nenhuma variante com convolução passa da logística. O `largo` é o análogo da logística treinado como a
   rede (Adam, parada antecipada), e fica abaixo dela (−0,040, IC −0,087 a +0,002). Tudo vai na direção do D2 (permutação
   gene → pixel sem efeito) e do bloco B do `05_eval`. O critério formal de descarte (BBC-CV pareada "melhor imagem − melhor
   tabular", PARTE C de `docs/relatorio_imagem.txt`) ainda não foi rodado.
5. **Cautela:** com GS = 46, um GS a mais ou a menos acertado muda o F1 do GS em 0,01 a 0,02. Diferenças de 0,02 a 0,05
   entre as variantes costumam estar no ruído. Só a do DRO contra o `profundo` sai do 0, e por pouco.